In [1]:
from langchain_google_genai import GoogleGenerativeAI
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = GoogleGenerativeAI(model="gemini-3.6-flash", google_api_key="API_KEY")

C:\Users\jesse\AppData\Local\Temp\ipykernel_14712\3830718084.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [2]:
loader = DirectoryLoader(
    "F:\LLM\knowledge_base",
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

docs = loader.load()
print(f"Loaded {len(docs)} markdown files")

Loaded 25 markdown files


In [3]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)

In [4]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

C:\Users\jesse\AppData\Local\Temp\ipykernel_14712\3409896792.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

In [6]:
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

In [7]:
print(vectorstore._collection.count())

125


In [8]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:

{context}

Question: {question}
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [9]:

response = rag_chain.invoke("What are the air line policies")
print(response)

c:\Users\jesse\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'models/gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the provided context, the airline policy detailed is the **Route Change Policy**:

* **General Rule:** Customers may change their destination before departure.
* **Charges:**
  * **Within same zone:** ₹500
  * **Different zone:** Fare difference + ₹1000
  * **International:** Fare difference + ₹2500
* **Restrictions:**
  * Changes are not permitted after check-in.
  * Maximum route changes allowed: 2 per booking.
